# Linly-Dubbing Colab WebUI
## 1. 环境准备 (Environment Setup)
基于第一性原理，我们需要首先确认硬件环境，然后安装操作系统级别的底层依赖，最后才是应用层的Python库。

In [19]:
# [Step 1.1] 硬件检查
# 确认分配到的 GPU 类型
!nvidia-smi

Thu Jan 29 08:12:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [20]:
# [Step 1.2] 获取代码
# 使用幂等性逻辑：如果目录已存在，则跳过 clone，避免重复执行报错
import os
if not os.path.exists('/content/Linly-Dubbing'):
    %cd /content/
    !git clone https://github.com/infinite-gaming-studio/Linly-Dubbing.git --depth 1
else:
    print("Project already cloned.")

%cd /content/Linly-Dubbing
!git submodule update --init --recursive

Project already cloned.
/content/Linly-Dubbing


In [21]:
# [Step 1.3] 安装系统级依赖 (System Dependencies)
# 优先安装系统库，确保 C++ 编译环境就绪
!apt-get update -qq
!apt-get install -y -qq build-essential libfst-dev ffmpeg espeak-ng libsndfile1 \
    libavfilter-dev libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev libswscale-dev libswresample-dev > /dev/null
print("System dependencies installed.")
!ffmpeg -version | head -n 1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
System dependencies installed.
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers


In [22]:
# [Step 1.4] Python 依赖安装 (Python Dependencies via uv)
# Colab 默认 Numpy 版本往往较高 (>=2.0)，而本项目依赖需要 Numpy < 2.0
# 我们使用 'uv' 进行极速安装，并强制处理冲突

# 1. 安装 uv
!pip install uv

# 2. 使用 uv 安装依赖 (比 pip 快 10-100 倍)
# 注意：我们添加 --system 标志以允许 uv 安装到 Colab 的系统 Python 环境中
print("Installing requirements with uv... This might take a minute but is much faster/safer than pip.")

# 强制重装 numpy 以确保版本正确
!uv pip install --system --force-reinstall "numpy<2.0.0" "setuptools"

# [CRITICAL] 强制重装 Torch 全家桶以确保版本兼容 (Fix: torch.library missing register_fake)
!uv pip install --system --force-reinstall torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1

# 1. 运行时修补 (Runtime Patching)
# 将 numpy==1.26.3 修改为 numpy<2.0.0 以避免与系统环境过分冲突，同时防止安装 2.0+
!sed -i 's/numpy==1.26.3/numpy<2.0.0/g' requirements.txt
print("Patched requirements.txt: Relaxed numpy constraint.")

# [CRITICAL] Patch TTS submodule for Python 3.12 support
# TTS officially requires <3.12, but Colab is 3.12. We patch setup.py to bypass check.
!sed -i 's/if Version(python_version) < Version("3.9") or Version(python_version) >= Version("3.12"):/# if Version(python_version) < Version("3.9") or Version(python_version) >= Version("3.12"):/g' submodules/TTS/setup.py
!sed -i 's/    raise RuntimeError("TTS requires python >= 3.9 and < 3.12 " "but your Python version is {}".format(sys.version))/#     raise RuntimeError("TTS requires python >= 3.9 and < 3.12 " "but your Python version is {}".format(sys.version))/g' submodules/TTS/setup.py
!sed -i 's/python_requires=">=3.9.0, <3.12",/python_requires=">=3.9.0, <3.13",/g' submodules/TTS/setup.py
print("Patched TTS submodule for Python 3.12 compatibility.")

# 2. 安装 uv
!pip install uv

# 3. 使用 uv 安装依赖 (比 pip 快 10-100 倍)
# 注意：我们添加 --system 标志以允许 uv 安装到 Colab 的系统 Python 环境中
print("Installing requirements with uv... This might take a minute but is much faster/safer than pip.")

# 强制重装 numpy 以确保版本正确
!uv pip install --system --force-reinstall "numpy<2.0.0" "setuptools"

# [CRITICAL] 强制重装 Torch 全家桶以确保版本兼容 (Fix: torch.library missing register_fake)
!uv pip install --system --force-reinstall torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1

# 安装项目依赖
!uv pip install --system -r requirements.txt
!uv pip install --system -r requirements_module.txt

print("Python dependencies installed successfully.")

Installing requirements with uv... This might take a minute but is much faster/safer than pip.
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 45ms                                          
Prepared 2 packages in 0.41ms                                            
Uninstalled 2 packages in 22ms
Installed 2 packages in 21ms                                
 ~ numpy==1.26.4
 ~ setuptools==80.10.2
Using Python 3.12.12 environment at: /usr
Resolved 25 packages in 108ms                                        
Prepared 25 packages in 977ms                                            
Uninstalled 25 packages in 365ms
Installed 25 packages in 267ms9.2.26                        
 ~ filelock==3.20.3
 - fsspec==2024.12.0
 + fsspec==2026.1.0
 ~ jinja2==3.1.6
 ~ markupsafe==3.0.3
 ~ mpmath==1.3.0
 - networkx==2.8.8
 + networkx==3.6.1
 - numpy==1.26.4
 + numpy==2.4.1
 ~ nvidia-cublas-cu12==12.1.3.1
 ~ nvidia-cuda-cupti-cu12==12.1.105
 ~ nvidia-cuda-nvrtc-cu12==12.1.105
 ~ nvidia-cuda-r

## 2. 资源获取 (Resource Acquisition)
下载运行所需的预训练模型文件。

In [23]:
# [Step 2.1] 下载模型
# 创建目录结构
!mkdir -p models/ASR/whisper

# 下载 wav2vec2 模型 (使用 -nc 参数避免重复下载)
!wget -nc https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth \
    -O models/ASR/whisper/wav2vec2_fairseq_base_ls960_asr_ls960.pth

# 运行项目自带的下载脚本下载其他模型
!python scripts/huggingface_download.py

File ‘models/ASR/whisper/wav2vec2_fairseq_base_ls960_asr_ls960.pth’ already there; not retrieving.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 18 files: 100% 18/18 [00:00<00:00, 1476.26it/s]
Fetching 12 files: 100% 12/12 [00:00<00:00, 1687.62it/s]
Fetching 10 files: 100% 10/10 [00:00<0

## 3. 应用启动 (Application Launch)
配置环境并启动 WebUI。

In [24]:
# [Step 3.1] 环境配置
%cd /content/Linly-Dubbing
!cp env.example .env
# 如果需要配置 API Key，请手动编辑 .env 文件或在 WebUI 界面中设置

/content/Linly-Dubbing


In [25]:
# [Step 3.2] 功能自检 (可选)
# 测试核心功能模块是否正常加载
!python -m tools.do_everything

2026-01-29 08:13:21.544120: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769674401.564225   21568 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769674401.570518   21568 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769674401.588553   21568 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769674401.588580   21568 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769674401.588584   21568 computation_placer.cc:177] computation placer alr

In [ ]:
# [Step 3.3] 启动 WebUI
# 启动后点击输出结果中的 public URL 即可访问
!python webui.py

2026-01-29 08:14:52.467027: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769674492.486306   21955 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769674492.492200   21955 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769674492.507317   21955 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769674492.507341   21955 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769674492.507347   21955 computation_placer.cc:177] computation placer alr